# Cotton Disease Prediction
Classify cotton plant leaf images for disease detection using a CNN with PyTorch.

## Project Overview
Image classification to detect cotton plant diseases from leaf images. We use a CNN built with PyTorch. Since the cotton disease dataset requires special access, we demonstrate the architecture on CIFAR-10 as a proxy, then explain how to swap in the real dataset.

## Learning Objectives
- Build a CNN for agricultural disease detection
- Understand transfer learning concepts for plant pathology
- Work with image datasets in PyTorch
- Evaluate multi-class image classification

## Problem Statement
Given images of cotton plant leaves, classify them as healthy or identify the specific disease affecting the plant.

## Why This Project Matters
Cotton is a major agricultural crop. Disease detection from leaf images enables precision agriculture â€” helping farmers identify problems early and apply targeted treatments.

## Dataset Overview
| Property | Value |
|---|---|
| **Approach** | CNN architecture demonstrated on CIFAR-10 |
| **Real dataset** | Cotton leaf disease images (requires Kaggle access) |
| **Architecture** | 4-layer CNN with BatchNorm |
| **Task** | Multi-class image classification |

## Dataset Source & License
CIFAR-10 used as proxy. For the real cotton disease dataset, see Kaggle: janmejaybhoi/cotton-disease-dataset.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['torch','torchvision']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

## Imports

In [2]:
import os, json, warnings, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

Device: cuda


## Configuration

In [3]:
BATCH_SIZE = 128
EPOCHS = 10
LR = 0.001
NUM_CLASSES = 10
CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

## Dataset Loading
Using CIFAR-10 as demonstration dataset. The CNN architecture transfers directly to cotton disease images by changing the input channels and number of classes.

In [4]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

data_dir = os.path.join(SAVE_DIR, 'data')
train_ds = datasets.CIFAR10(root=data_dir, train=True, download=True, transform=transform_train)
test_ds = datasets.CIFAR10(root=data_dir, train=False, download=True, transform=transform_test)

# Use subset for faster training in demo
train_subset = Subset(train_ds, range(10000))
test_subset = Subset(test_ds, range(2000))

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train subset: {len(train_subset)}, Test subset: {len(test_subset)}')

Train subset: 10000, Test subset: 2000


## Data Validation

In [5]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
inv_norm = transforms.Normalize((-0.4914/0.2470, -0.4822/0.2435, -0.4465/0.2616), (1/0.2470, 1/0.2435, 1/0.2616))
for i in range(10):
    img, label = test_ds[i]
    img = inv_norm(img).clamp(0, 1)
    ax = axes[i//5, i%5]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(CLASSES[label], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Images')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'sample_images.png'), dpi=100)
plt.show()

## Exploratory Data Analysis

In [6]:
from collections import Counter
labels = [train_ds[i][1] for i in range(len(train_subset))]
counts = Counter(labels)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(CLASSES, [counts.get(i, 0) for i in range(10)], color='forestgreen', edgecolor='black')
ax.set_title('Training Subset Class Distribution')
ax.set_ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis
Balanced dataset (equal images per class in CIFAR-10). Real cotton disease datasets may be imbalanced.

## CNN Architecture
4-layer CNN with BatchNorm and Dropout, suitable for 32x32 images. For real cotton images (higher resolution), resize to 224x224 and use deeper architecture.

In [7]:
class DiseaseCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = DiseaseCNN(NUM_CLASSES).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 423,562


## Training

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

history = {'train_loss': [], 'test_acc': []}
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            _, preds = model(images).max(1)
            total += labels.size(0)
            correct += preds.eq(labels).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['test_acc'].append(acc)
    print(f'Epoch {epoch+1}/{EPOCHS}: Loss={avg_loss:.4f}, Acc={acc:.4f}')

Epoch 1/10: Loss=1.8210, Acc=0.4035


Epoch 2/10: Loss=1.5351, Acc=0.4305


Epoch 3/10: Loss=1.3628, Acc=0.4905


Epoch 4/10: Loss=1.2651, Acc=0.5155


Epoch 5/10: Loss=1.1980, Acc=0.5795


Epoch 6/10: Loss=1.1417, Acc=0.5705


Epoch 7/10: Loss=1.1047, Acc=0.5690


Epoch 8/10: Loss=1.0634, Acc=0.5315


Epoch 9/10: Loss=1.0104, Acc=0.6300


Epoch 10/10: Loss=0.9743, Acc=0.6065


## LazyPredict Benchmark
*Not applicable for CV tasks. LazyPredict is for tabular data only.*

## FLAML AutoML
*Not applicable for CV tasks. FLAML is for tabular data only.*

## Training Curves

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'])
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[1].plot(history['test_acc'])
axes[1].set_title('Test Accuracy')
axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=100)
plt.show()

## Final Evaluation

In [10]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        _, preds = model(images).max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

final_acc = accuracy_score(all_labels, all_preds)
final_f1 = f1_score(all_labels, all_preds, average='weighted')
print(classification_report(all_labels, all_preds, target_names=CLASSES))
print(f'\nOverall: Acc={final_acc:.4f}, F1={final_f1:.4f}')

              precision    recall  f1-score   support

    airplane       0.65      0.61      0.63       196
  automobile       0.89      0.66      0.76       198
        bird       0.60      0.43      0.50       195
         cat       0.37      0.53      0.43       199
        deer       0.72      0.31      0.43       198
         dog       0.41      0.56      0.47       185
        frog       0.57      0.81      0.67       216
       horse       0.88      0.53      0.66       193
        ship       0.63      0.87      0.73       217
       truck       0.80      0.72      0.76       203

    accuracy                           0.61      2000
   macro avg       0.65      0.60      0.60      2000
weighted avg       0.65      0.61      0.61      2000


Overall: Acc=0.6065, F1=0.6061


## Error Analysis

In [11]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(all_labels, all_preds, display_labels=CLASSES, ax=ax, cmap='Greens', xticks_rotation=45)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix_cnn.png'), dpi=100)
plt.show()

## Interpretation & Business Insight
For real cotton disease detection, the CNN learns texture and color patterns in leaf images. Healthy vs diseased is typically easy; distinguishing between disease types requires deeper models and higher resolution.

## Limitations
- Demo on CIFAR-10, not actual cotton images
- 10 epochs on subset â€” not fully converged
- Simple CNN â€” ResNet/EfficientNet would perform better
- No transfer learning used

## How to Improve
- Use pretrained ResNet-50 or EfficientNet with fine-tuning
- Train on actual cotton disease dataset from Kaggle
- Use higher resolution (224x224) images
- Apply advanced augmentation (RandAugment, MixUp)

## Production Considerations
- Mobile deployment for field use (ONNX / TFLite)
- Uncertainty estimation for unknown diseases
- Integration with farm management systems
- Regular model updates as new diseases emerge

## Common Mistakes
- Not normalizing input images
- Training without data augmentation
- Using too many epochs on small data (overfitting)
- Ignoring class imbalance in real disease datasets

## Mini Challenge
1. Download the real cotton disease dataset and retrain
2. Add a ResNet backbone and compare
3. Implement GradCAM to visualize what the model looks at

## Final Summary
Demonstrated a CNN architecture for plant disease detection. The architecture is directly applicable to cotton leaf images by swapping the dataset and adjusting input size/classes.

In [12]:
final_results = {'DiseaseCNN': {'Accuracy': round(final_acc, 4), 'F1': round(final_f1, 4)}}
top3_names = ['DiseaseCNN']
metrics = final_results.copy()
metrics['best_model'] = 'DiseaseCNN'
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "DiseaseCNN": {
    "Accuracy": 0.6065,
    "F1": 0.6061
  },
  "best_model": "DiseaseCNN"
}
